# WSMTE — 02_feature_engineering.ipynb
**LOCAL PC** | Steps 9–17 from DATA_PIPELINE.md  
Wavelet denoising → technical indicators → split → scale → sliding windows → targets.

**Prerequisites**: `data/processed/merged_data.csv` must exist (run `01_data_prep.ipynb` first).

In [1]:
import sys, os

# Find project root by walking up until config/config.py is found
_search = os.path.abspath('')
while not os.path.exists(os.path.join(_search, 'config', 'config.py')):
    _parent = os.path.dirname(_search)
    if _parent == _search:
        raise RuntimeError("Cannot find project root — is config/config.py missing?")
    _search = _parent

PROJECT_ROOT = _search
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import pywt
import joblib
import json
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from config.config import CONFIG

# Make every CONFIG path absolute so no cell depends on CWD
_path_keys = [k for k in CONFIG if isinstance(CONFIG[k], str) and
              (k.endswith('_dir') or k.endswith('_file') or k == 'ablation_results')]
for k in _path_keys:
    CONFIG[k] = os.path.join(PROJECT_ROOT, CONFIG[k])

print(f"Project root  : {PROJECT_ROOT}")
print(f"processed_dir : {CONFIG['processed_data_dir']}")
print(f"merged_data exists: {os.path.exists(os.path.join(CONFIG['processed_data_dir'], 'merged_data.csv'))}")

Project root  : d:\WSMTE
processed_dir : d:\WSMTE\data/processed/
merged_data exists: True


## Load merged_data.csv

In [2]:
df = pd.read_csv(CONFIG['processed_data_dir'] + 'merged_data.csv')
df['date'] = pd.to_datetime(df['date']).dt.date
df = df.sort_values('date').reset_index(drop=True)
print(f'Loaded merged_data.csv: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Columns: {list(df.columns)}')

Loaded merged_data.csv: (1092, 12)
Date range: 2020-01-01 to 2024-05-31
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'date', 'polarity_company', 'polarity_company_max', 'subjectivity', 'polarity_market', 'polarity_market_max']


## Step 9 — Coif3 Wavelet Denoising
> Applied to Open, High, Low, Close, Volume independently BEFORE any indicator computation.

In [3]:
def coif3_denoise(series):
    """
    Coif3, level=1, soft thresholding.
    Universal threshold via median absolute deviation.
    Returns denoised array of same length as input.
    """
    coeffs = pywt.wavedec(series, CONFIG['wavelet'], level=CONFIG['wavelet_level'])
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(len(series)))
    coeffs[1:] = [pywt.threshold(c, threshold, mode=CONFIG['wavelet_mode'])
                  for c in coeffs[1:]]
    return pywt.waverec(coeffs, CONFIG['wavelet'])[:len(series)]

for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[f'{col}_d'] = coif3_denoise(df[col].values.copy())
    print(f'Denoised {col}: std_orig={df[col].std():.4f}  std_denoised={df[f"{col}_d"].std():.4f}')

Denoised Open: std_orig=3493.9915  std_denoised=3493.2675
Denoised High: std_orig=3490.4169  std_denoised=3489.9999
Denoised Low: std_orig=3499.9233  std_denoised=3499.3597
Denoised Close: std_orig=3496.5883  std_denoised=3495.9430
Denoised Volume: std_orig=212165.5473  std_denoised=203304.0366


## Step 10 — Technical Indicators on DENOISED Prices
> All indicators use `_d` columns, NOT raw prices.

In [4]:
# ── RSI(14) ──────────────────────────────────────────────────────
def compute_rsi(series, period=None):
    if period is None:
        period = CONFIG['rsi_period']
    delta = series.diff()
    gain  = delta.where(delta > 0, 0.0).rolling(period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI_d'] = compute_rsi(df['Close_d'])
print(f'RSI_d range: {df["RSI_d"].min():.1f} to {df["RSI_d"].max():.1f}')

RSI_d range: 3.9 to 100.0


In [5]:
# ── MACD histogram: EMA(12) - EMA(26) ──────────────────────────────────
ema_fast = df['Close_d'].ewm(span=CONFIG['ema_fast'], adjust=False).mean()
ema_slow = df['Close_d'].ewm(span=CONFIG['ema_slow'], adjust=False).mean()
df['MACD_d'] = ema_fast - ema_slow
print(f'MACD_d range: {df["MACD_d"].min():.4f} to {df["MACD_d"].max():.4f}')

MACD_d range: -998.7287 to 447.9685


In [6]:
# ── Bollinger Band Width: (Upper - Lower) / Middle ───────────────────────
sma20 = df['Close_d'].rolling(CONFIG['bb_period']).mean()
std20 = df['Close_d'].rolling(CONFIG['bb_period']).std()
df['BB_width_d'] = (CONFIG['bb_std'] * 2 * std20) / sma20
print(f'BB_width_d range: {df["BB_width_d"].min():.4f} to {df["BB_width_d"].max():.4f}')

BB_width_d range: 0.0147 to 0.5518


In [7]:
# ── ROC(5): 5-day rate of change ───────────────────────────────────────
df['ROC_d'] = df['Close_d'].pct_change(periods=CONFIG['roc_period']) * 100
print(f'ROC_d range: {df["ROC_d"].min():.4f} to {df["ROC_d"].max():.4f}')

ROC_d range: -17.6639 to 11.1385


## Step 11 — Build Final Feature Dataframe
> Drop warmup rows (26 days for MACD EMA(26) convergence).

In [8]:
FEATURE_COLUMNS = CONFIG['feature_columns']
# Expected: ['Close_d','Volume_d','RSI_d','MACD_d','BB_width_d','ROC_d',
#            'polarity_company','polarity_market','subjectivity']

df = df.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
print(f'After warmup drop ({CONFIG["warmup_days"]} days): {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')

df[['date'] + FEATURE_COLUMNS + ['Close']].to_csv(
    CONFIG['processed_data_dir'] + 'final_dataset.csv', index=False)
print('Saved → data/processed/final_dataset.csv')

After warmup drop (26 days): (1073, 21)
Date range: 2020-01-28 to 2024-05-31
Saved → data/processed/final_dataset.csv


## Step 12 — Verify No Missing Values

In [9]:
assert df[FEATURE_COLUMNS].isnull().sum().sum() == 0, \
    'Missing values found — fix pipeline before continuing'
print(f'No missing values ✓  |  Final shape: {df.shape}')
print(df[FEATURE_COLUMNS].describe().round(4))

No missing values ✓  |  Final shape: (1073, 21)
          Close_d      High_d       Low_d      Open_d      Volume_d  \
count   1073.0000   1073.0000   1073.0000   1073.0000  1.073000e+03   
mean   16471.4190  16561.8806  16373.0369  16481.2562  3.955784e+05   
std     3481.6012   3474.8326   3486.1910   3478.9811  2.037428e+05   
min     7651.2707   8166.3622   7638.6258   7815.1455 -1.370970e+04   
25%    14610.3574  14742.6884  14506.3885  14660.4402  2.510780e+05   
50%    17223.3091  17325.8992  17109.9200  17247.4807  3.170064e+05   
75%    18402.3248  18480.9368  18317.6354  18415.5419  5.203067e+05   
max    22997.1633  23090.8931  22899.9755  22997.7953  1.795983e+06   

           RSI_d     MACD_d  BB_width_d      ROC_d  polarity_company  \
count  1073.0000  1073.0000   1073.0000  1073.0000         1073.0000   
mean     58.2919    66.7751      0.0728     0.3265            0.2027   
std      22.2169   199.6392      0.0611     2.6091            0.4330   
min       3.8873  -998.7

## Step 13 — Temporal Split (70 / 15 / 15, NO shuffling)

In [10]:
n = len(df)
train_end = int(n * CONFIG['train_ratio'])
val_end   = int(n * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

train_df = df.iloc[:train_end].reset_index(drop=True)
val_df   = df.iloc[train_end:val_end].reset_index(drop=True)
test_df  = df.iloc[val_end:].reset_index(drop=True)

print(f'Total: {n}  |  Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')
assert train_df['date'].max() < val_df['date'].min(), 'Temporal leakage: train/val overlap!'
assert val_df['date'].max() < test_df['date'].min(),  'Temporal leakage: val/test overlap!'
print(f'Train: {train_df["date"].min()} → {train_df["date"].max()}')
print(f'Val:   {val_df["date"].min()} → {val_df["date"].max()}')
print(f'Test:  {test_df["date"].min()} → {test_df["date"].max()}')
print('No temporal leakage ✓')

Total: 1073  |  Train: 804  |  Val: 108  |  Test: 161
Train: 2020-01-28 → 2023-04-26
Val:   2023-04-27 → 2023-09-29
Test:  2023-10-03 → 2024-05-31
No temporal leakage ✓


## Step 14 — MinMaxScaler (fit on train ONLY)

In [11]:
scaler = MinMaxScaler()  # scales each feature independently

train_scaled = scaler.fit_transform(train_df[FEATURE_COLUMNS])
val_scaled   = scaler.transform(val_df[FEATURE_COLUMNS])
test_scaled  = scaler.transform(test_df[FEATURE_COLUMNS])

# Sanity check: train must be [0,1] since scaler was fit on it
assert train_scaled.max() <= 1.0 and train_scaled.min() >= 0.0, 'Scaler sanity check failed!'
print(f'Train scaled: min={train_scaled.min():.4f}  max={train_scaled.max():.4f}  ✓')
print(f'Val   scaled: min={val_scaled.min():.4f}  max={val_scaled.max():.4f}')
print(f'Test  scaled: min={test_scaled.min():.4f}  max={test_scaled.max():.4f}')

joblib.dump(scaler, CONFIG['processed_data_dir'] + 'scaler.pkl')
print('Saved → data/processed/scaler.pkl')

Train scaled: min=0.0000  max=1.0000  ✓
Val   scaled: min=0.0002  max=1.1236
Test  scaled: min=0.0003  max=1.3929
Saved → data/processed/scaler.pkl


## Step 15 — Sliding Windows (window_size=5)

In [12]:
WINDOW_SIZE = CONFIG['window_size']

def create_windows(scaled_data, raw_close, window_size=WINDOW_SIZE):
    """
    For each position i in [window_size, len(scaled_data)):
      X[i]     = scaled_data[i-window_size : i]        shape [5, 9]
      y_clf[i] = 1 if raw_close[i] > raw_close[i-1] else 0
      y_reg[i] = scaled_data[i][0]                     (scaled Close_d)
    """
    X, y_clf, y_reg = [], [], []
    for i in range(window_size, len(scaled_data)):
        X.append(scaled_data[i - window_size : i])
        y_clf.append(1 if raw_close[i] > raw_close[i - 1] else 0)
        y_reg.append(scaled_data[i][0])  # index 0 = Close_d
    return (np.array(X, dtype=np.float32),
            np.array(y_clf, dtype=np.int32),
            np.array(y_reg, dtype=np.float32))

X_train, y_clf_train, y_reg_train = create_windows(train_scaled, train_df['Close'].values)
X_val,   y_clf_val,   y_reg_val   = create_windows(val_scaled,   val_df['Close'].values)
X_test,  y_clf_test,  y_reg_test  = create_windows(test_scaled,  test_df['Close'].values)

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')
assert X_train.shape[1] == WINDOW_SIZE and X_train.shape[2] == CONFIG['n_features'], \
    f'Expected window shape [{WINDOW_SIZE}, {CONFIG["n_features"]}], got {X_train.shape[1:]}'
print(f'Window shape [{WINDOW_SIZE}, {CONFIG["n_features"]}] ✓')

X_train: (799, 5, 14)  |  X_val: (103, 5, 14)  |  X_test: (156, 5, 14)
Window shape [5, 14] ✓


In [13]:
# Save all arrays
arrays = {
    'X_train': X_train,     'X_val': X_val,     'X_test': X_test,
    'y_clf_train': y_clf_train, 'y_clf_val': y_clf_val, 'y_clf_test': y_clf_test,
    'y_reg_train': y_reg_train, 'y_reg_val': y_reg_val, 'y_reg_test': y_reg_test,
}
for name, arr in arrays.items():
    np.save(CONFIG['processed_data_dir'] + f'{name}.npy', arr)
    print(f'Saved {name}.npy  shape={arr.shape}  dtype={arr.dtype}')

Saved X_train.npy  shape=(799, 5, 14)  dtype=float32
Saved X_val.npy  shape=(103, 5, 14)  dtype=float32
Saved X_test.npy  shape=(156, 5, 14)  dtype=float32
Saved y_clf_train.npy  shape=(799,)  dtype=int32
Saved y_clf_val.npy  shape=(103,)  dtype=int32
Saved y_clf_test.npy  shape=(156,)  dtype=int32
Saved y_reg_train.npy  shape=(799,)  dtype=float32
Saved y_reg_val.npy  shape=(103,)  dtype=float32
Saved y_reg_test.npy  shape=(156,)  dtype=float32


## Step 16 — Class Imbalance Check

In [14]:
counts = Counter(y_clf_train)
ratio    = max(counts.values()) / min(counts.values())
up_pct   = counts[1] / len(y_clf_train) * 100
down_pct = counts[0] / len(y_clf_train) * 100
print(f'Up: {up_pct:.1f}%  |  Down: {down_pct:.1f}%  |  Ratio: {ratio:.2f}')

if ratio > CONFIG['imbalance_threshold']:  # > 1.5 (60:40)
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_clf_train)
    class_weight_dict = {0: float(weights[0]), 1: float(weights[1])}
    print(f'Imbalanced — applying class weights: {class_weight_dict}')
else:
    class_weight_dict = None
    print('Balanced — no class weighting needed')

with open(CONFIG['processed_data_dir'] + 'class_weights.json', 'w') as f:
    json.dump(class_weight_dict, f)
print('Saved → data/processed/class_weights.json')

Up: 54.4%  |  Down: 45.6%  |  Ratio: 1.20
Balanced — no class weighting needed
Saved → data/processed/class_weights.json


## Summary

In [15]:
print('=' * 55)
print('Feature engineering complete.')
print(f'  final_dataset.csv : {df.shape}')
print(f'  X_train           : {X_train.shape}')
print(f'  X_val             : {X_val.shape}')
print(f'  X_test            : {X_test.shape}')
print(f'  y_clf_train       : {y_clf_train.shape}  labels={set(y_clf_train)}')
print(f'  class_weights     : {class_weight_dict}')
print('=' * 55)
print('Ready for Day 3 — model implementation.')

Feature engineering complete.
  final_dataset.csv : (1073, 21)
  X_train           : (799, 5, 14)
  X_val             : (103, 5, 14)
  X_test            : (156, 5, 14)
  y_clf_train       : (799,)  labels={np.int32(0), np.int32(1)}
  class_weights     : None
Ready for Day 3 — model implementation.
